# L6c: Iterative Methods for Solving Linear Algebraic Equations

In this lecture, we will develop iterative methods for solving square systems of linear algebraic equations. These methods begin with an initial guess and use repeated corrections to approach a solution, offering an alternative when a direct solution requires too much computation or memory.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> - **Construct an iterative method:** Use a matrix splitting to derive an update from the residual, and distinguish successful convergence from stopping at an iteration limit.
> - **Assess convergence:** Explain the spectral-radius condition, identify when strict diagonal dominance guarantees convergence, and use an error bound to estimate a sufficient iteration count.
> - **Compare specific methods:** Explain how Jacobi, Gauss–Seidel, and successive over-relaxation construct their updates, and how these choices affect the work per iteration and convergence.

We will begin with the steps shared by these methods, derive the correction from a matrix splitting, and examine the conditions under which repeated corrections approach the solution. We will then use this framework to compare the three methods and interpret their behavior in the companion example.
___


## Examples
We will use the following example to compare iterative solutions with a direct solution and examine their computational cost:

> [▶ Fun with Iterative Methods](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb). How do repeated corrections compare with a direct solution? We solve a randomly generated linear system using a selected iterative method, compare its result with Julia's direct solution, and benchmark runtime and memory use. Changing the selected method allows us to explore Jacobi, Gauss–Seidel, and successive over-relaxation.

___


## General iterative method
Suppose we have a square system of linear equations given by:

$$
\mathbf{A}\mathbf{x}=\mathbf{b},
$$

where $\mathbf{A}\in\mathbb{R}^{n\times n}$ is the system matrix, $\mathbf{x}\in\mathbb{R}^{n}$ contains the $n$ unknowns, and $\mathbf{b}\in\mathbb{R}^{n}$ is the right-hand side vector. We assume that $\mathbf{A}$ is nonsingular, so the system has a unique solution, and seek a sequence of approximations that approaches this solution.

Starting from an initial guess $\mathbf{x}^{(0)}$, we measure how well the current approximation satisfies the equations using the residual vector:

$$
\mathbf{r}^{(k)}=\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)},
$$

where $k$ counts the corrections already made. Each residual component measures the mismatch in one equation; the residual is zero at the exact solution.

To construct a correction, choose a splitting $\mathbf{A}=\mathbf{M}-\mathbf{N}$, where $\mathbf{M}$ is nonsingular and chosen so that systems with $\mathbf{M}$ are inexpensive to solve, and $\mathbf{N}=\mathbf{M}-\mathbf{A}$. A diagonal or triangular $\mathbf{M}$ is a common choice. Keeping this splitting fixed across iterations gives a _stationary iterative method_.

Let's sketch the steps before deriving the correction.

__Initialize__: Choose an initial guess $\mathbf{x}^{(0)}$, an absolute residual tolerance $\epsilon>0$, and a maximum number of corrections $\texttt{maxiter}$. Set the iteration counter $k\gets0$ and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:

1. Calculate the residual vector $\mathbf{r}^{(k)}\gets\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)}$.
2. Check whether to stop:
   - If $\|\mathbf{r}^{(k)}\|_2<\epsilon$, set $\texttt{converged}\gets\texttt{true}$ and return $\mathbf{x}^{(k)}$ with this status.
   - Otherwise, if $k\geq\texttt{maxiter}$, return $\mathbf{x}^{(k)}$ with $\texttt{converged}=\texttt{false}$ and a warning that the residual tolerance was not met.
3. Calculate the correction $\mathbf{d}^{(k)}$ by solving $\mathbf{M}\mathbf{d}^{(k)}=\mathbf{r}^{(k)}$.
4. Update the approximation: $\mathbf{x}^{(k+1)}\gets\mathbf{x}^{(k)}+\mathbf{d}^{(k)}$.
5. Increment the iteration counter: $k\gets k+1$.

The correction can be written as $\mathbf{d}^{(k)}=\mathbf{M}^{-1}\mathbf{r}^{(k)}$, but computing it requires solving a system with $\mathbf{M}$; we do not need to form an explicit inverse. The choice of $\mathbf{M}$ affects both the work needed for each correction and whether the iteration converges.

A small residual means that the equations are nearly satisfied. How close the approximation is to the true solution also depends on the sensitivity of the system, so a residual tolerance is not automatically a bound on the solution error. Next, we will derive the correction from the matrix splitting and show how the choice of $\mathbf{M}$ connects the specific methods.

___


## Update direction
Where does the correction $\mathbf{d}^{(k)}$ come from? Starting from the original system and substituting the splitting $\mathbf{A}=\mathbf{M}-\mathbf{N}$ gives:

$$
\begin{align*}
\mathbf{A}\mathbf{x}&=\mathbf{b},\\
(\mathbf{M}-\mathbf{N})\mathbf{x}&=\mathbf{b},\\
\mathbf{M}\mathbf{x}&=\mathbf{b}+\mathbf{N}\mathbf{x}.
\end{align*}
$$

To turn this identity into an iteration, we evaluate the right-hand side using the current approximation $\mathbf{x}^{(k)}$ and solve for the next approximation $\mathbf{x}^{(k+1)}$. Since $\mathbf{M}$ is nonsingular, this defines the update as:

$$
\begin{align*}
\mathbf{M}\mathbf{x}^{(k+1)}&=\mathbf{b}+\mathbf{N}\mathbf{x}^{(k)},\\
\mathbf{x}^{(k+1)}&=\mathbf{M}^{-1}\mathbf{b}+\mathbf{M}^{-1}\mathbf{N}\mathbf{x}^{(k)}.
\end{align*}
$$

Now let's express this update as a correction to the current approximation. Using $\mathbf{N}=\mathbf{M}-\mathbf{A}$, with $\mathbf{I}$ denoting the $n\times n$ identity matrix, we obtain:

$$
\mathbf{M}^{-1}\mathbf{N}
=\mathbf{M}^{-1}(\mathbf{M}-\mathbf{A})
=\mathbf{I}-\mathbf{M}^{-1}\mathbf{A}.
$$

Substituting this identity into the update gives:

$$
\begin{align*}
\mathbf{x}^{(k+1)}
&=\mathbf{M}^{-1}\mathbf{b}+(\mathbf{I}-\mathbf{M}^{-1}\mathbf{A})\mathbf{x}^{(k)}\\
&=\mathbf{x}^{(k)}+\mathbf{M}^{-1}\underbrace{(\mathbf{b}-\mathbf{A}\mathbf{x}^{(k)})}_{\text{residual }\mathbf{r}^{(k)}}\\
&=\mathbf{x}^{(k)}+\underbrace{\mathbf{M}^{-1}\mathbf{r}^{(k)}}_{\text{correction }\mathbf{d}^{(k)}}.
\end{align*}
$$

The correction therefore solves $\mathbf{M}\mathbf{d}^{(k)}=\mathbf{r}^{(k)}$, which is the step used in our general algorithm. Different choices of $\mathbf{M}$ produce the Jacobi, Gauss–Seidel, and successive over-relaxation methods; they determine how the residual is converted into a correction.

> __Stationary iteration:__
>
> Define the _iteration matrix_ $\mathbf{G}$ and the constant vector $\mathbf{c}$ by:
>
> $$
> \mathbf{G}=\mathbf{M}^{-1}\mathbf{N},
> \qquad
> \mathbf{c}=\mathbf{M}^{-1}\mathbf{b}.
> $$
>
> The same update can then be written as:
>
> $$
> \mathbf{x}^{(k+1)}=\mathbf{G}\mathbf{x}^{(k)}+\mathbf{c}.
> $$
>
> The matrix and vector remain fixed during the iteration. An exact solution is unchanged by this update because its residual is zero, but this alone does not establish that the iterates approach it.

We next examine how the iteration matrix determines whether approximations starting from an arbitrary initial guess converge to the solution.

___


## Convergence
When do repeated corrections approach the true solution $\mathbf{x}^{\star}$? Define the solution error by $\mathbf{e}^{(k)}=\mathbf{x}^{(k)}-\mathbf{x}^{\star}$. Subtracting the fixed-point equation from the update gives:

$$
\begin{aligned}
\mathbf{e}^{(k+1)}
&=(\mathbf{G}\mathbf{x}^{(k)}+\mathbf{c})-(\mathbf{G}\mathbf{x}^{\star}+\mathbf{c})
=\mathbf{G}\mathbf{e}^{(k)},\\
\mathbf{e}^{(k)}&=\mathbf{G}^{k}\mathbf{e}^{(0)}.
\end{aligned}
$$

Thus, convergence from every initial guess requires the powers of the iteration matrix to approach zero.

> __Spectral-radius convergence condition:__
>
> For the stationary iteration defined above, let $\lambda_i$ denote the eigenvalues of $\mathbf{G}$. The iteration converges to $\mathbf{x}^{\star}$ from every initial guess if and only if:
>
> $$
> \rho(\mathbf{G})=\max_i|\lambda_i|<1.
> $$
>
> The spectral radius measures the largest eigenvalue magnitude. This condition guarantees eventual decay of every initial error, but the error need not decrease at every step.

The [advanced convergence notebook](CHEME-5800-L6c-Advanced-Convergence-IterativeMethods-Fall-2026.ipynb) develops the error recurrence in more detail. Next, we examine a sufficient condition that can be checked directly from the entries of $\mathbf{A}$.

### Diagonal dominance

A matrix is _strictly diagonally dominant by rows_ when the magnitude of every diagonal entry exceeds the sum of the magnitudes of the other entries in its row:

$$
|a_{ii}|>\sum_{j\ne i}|a_{ij}|,\qquad i=1,\ldots,n.
$$

This condition guarantees convergence of both Jacobi and Gauss–Seidel from every initial guess. It is sufficient, but not necessary.

To see why it works for Jacobi, let $\mathbf{D}$ contain the diagonal entries of $\mathbf{A}$. The Jacobi splitting uses $\mathbf{M}=\mathbf{D}$ and $\mathbf{N}=\mathbf{D}-\mathbf{A}$, so the off-diagonal entries of $\mathbf{N}$ have the opposite signs to those of $\mathbf{A}$. For the Jacobi iteration matrix $\mathbf{G}_{J}=\mathbf{D}^{-1}(\mathbf{D}-\mathbf{A})$, the induced infinity-norm, the largest absolute row sum, gives:

$$
\|\mathbf{G}_{J}\|_\infty
=\max_i\frac{\sum_{j\ne i}|a_{ij}|}{|a_{ii}|}<1.
$$

For any eigenpair $(\lambda,\mathbf{v})$ with $\mathbf{v}\ne\mathbf{0}$, the norm inequality implies:

$$
\begin{aligned}
|\lambda|\,\|\mathbf{v}\|_\infty
&=\|\mathbf{G}_{J}\mathbf{v}\|_\infty
\le\|\mathbf{G}_{J}\|_\infty\|\mathbf{v}\|_\infty,\\
\rho(\mathbf{G}_{J})&\le\|\mathbf{G}_{J}\|_\infty<1.
\end{aligned}
$$

This proves the Jacobi result. Gauss–Seidel uses a triangular splitting and requires a separate argument; the Jacobi row-sum formula does not apply to its iteration matrix.

### Rate of convergence and an iteration bound

The spectral-radius condition establishes convergence. To bound the error after a finite number of corrections, suppose an induced matrix norm satisfies $q=\|\mathbf{G}\|<1$. Using its corresponding vector norm gives:

$$
\|\mathbf{e}^{(k)}\|
=\|\mathbf{G}^{k}\mathbf{e}^{(0)}\|
\le\|\mathbf{G}^{k}\|\,\|\mathbf{e}^{(0)}\|
\le q^{k}\|\mathbf{e}^{(0)}\|.
$$

Let $E_0>0$ be a known upper bound on the initial error, and choose a solution-error tolerance $0<\epsilon_e<E_0$. For $0<q<1$, requiring $q^kE_0\le\epsilon_e$ yields:

$$
\begin{aligned}
k\ln q&\le\ln(\epsilon_e/E_0),\\
k&\ge\left\lceil\frac{\ln(E_0/\epsilon_e)}{-\ln q}\right\rceil.
\end{aligned}
$$

Division reverses the inequality because $\ln q<0$; the ceiling rounds up to a whole number of corrections. For example, $q=0.5$, $E_0=1$, and $\epsilon_e=10^{-3}$ give a sufficient count of $10$ corrections. A smaller contraction factor gives a smaller bound.

If $E_0\le\epsilon_e$, the initial guess already meets the error bound; if $q=0$, one correction is exact. A chosen norm with $q\ge1$ gives no decay guarantee from this bound, even when the spectral radius is below one. The error tolerance $\epsilon_e$ is distinct from the residual tolerance $\epsilon$ used in our algorithm, and the initial error bound may be unavailable in practice.

___


## Specific Methods
The choice of $\mathbf{M}$ determines how we compute each correction. Write $\mathbf{A}=\mathbf{D}+\mathbf{L}+\mathbf{U}$, where $\mathbf{D}$ is diagonal and $\mathbf{L}$ and $\mathbf{U}$ contain the strictly lower and upper triangular entries of $\mathbf{A}$. With nonzero diagonal entries and a relaxation factor $\omega>0$, the three methods use:

$$
\begin{aligned}
\mathbf{M}_{J}&=\mathbf{D}, &&\text{Jacobi},\\
\mathbf{M}_{GS}&=\mathbf{D}+\mathbf{L}, &&\text{Gauss–Seidel},\\
\mathbf{M}_{\omega}&=\frac{1}{\omega}\mathbf{D}+\mathbf{L}, &&\text{successive over-relaxation}.
\end{aligned}
$$

* __Jacobi method__: A diagonal solve updates each unknown using values from the previous iteration. These updates can be computed in parallel. Strict row diagonal dominance guarantees convergence. [Explore the Jacobi algorithm.](CHEME-5800-L6c-Algorithm-JacobiMethod-Fall-2026.ipynb)

* __Gauss–Seidel method__: A lower triangular solve updates unknowns in sequence, using new values as soon as they are available. This can reduce the iteration count, but it introduces dependencies between updates; it is not universally faster than Jacobi. Strict row diagonal dominance also guarantees convergence. [Explore the Gauss–Seidel algorithm.](CHEME-5800-L6c-Algorithm-GaussSeidel-Fall-2026.ipynb)

* __Successive over-relaxation (SOR) method__: The relaxation factor $\omega$ scales each component's correction from a Gauss–Seidel-style update. Values $0<\omega<1$ give under-relaxation, $\omega=1$ recovers Gauss–Seidel, and $1<\omega<2$ gives over-relaxation. If $\mathbf{A}$ is symmetric positive definite, [convergence is guaranteed for $0<\omega<2$](https://netlib.org/linalg/html_templates/node16.html); strict diagonal dominance alone does not guarantee convergence over this entire interval. A suitable factor can reduce the iteration count, but the best choice depends on the matrix. [Explore the relaxation algorithm.](CHEME-5800-L6c-Algorithm-SOR-Fall-2026.ipynb)

For dense matrices, a residual calculation and a full correction sweep require work proportional to $n^2$ for each method; sparse implementations can use only the stored nonzero entries. Runtime therefore depends on both the cost of each sweep and the number of sweeps needed to meet the tolerance.

> __Example__
>
> [▶ Fun with Iterative Methods](CHEME-5800-L6c-Example-FunWithIterativeSolvers-Fall-2026.ipynb). Compare a selected iterative method with a direct solution, then benchmark runtime and memory use on the same randomly generated system.

Use the residual to assess whether a run reaches its tolerance before interpreting the timing comparison. A convergence theorem applies only when the generated matrix and selected method satisfy its assumptions.

___


## Lab
In [L6d](../L6d/CHEME-5800-L6d-Lab-IterativeLinearSolvers-Fall-2026.ipynb), we will compare Jacobi, Gauss–Seidel, and SOR on a diagonally dominant linear system using residual histories, iteration counts, and a direct solution as a reference.

___


## Summary
We developed stationary iterative methods by converting the residual into a correction through a matrix splitting.

> __Key Takeaways:__
>
> - **Residual-based corrections:** We derived a common update rule for the three methods. The splitting determines how we solve for each correction, while the residual measures how well the current approximation satisfies the equations. Reaching an iteration limit does not establish convergence.
> - **Convergence and error bounds:** We used the iteration matrix to characterize convergence from every initial guess. Strict row diagonal dominance provides a sufficient condition for Jacobi and Gauss–Seidel, and a contracting matrix norm lets us estimate a sufficient iteration count when an initial error bound is available.
> - **Choosing a method:** We compared independent Jacobi updates, sequential Gauss–Seidel updates, and relaxed updates. These choices affect both computational work and convergence; relaxation can help, but its benefit and convergence depend on the matrix and parameter choice.

The companion example and lab let us compare these predictions with computed residuals, iteration counts, and runtimes.

___
